# 06_speculative_decoding_simulation: Speculative Decoding Simulator

This notebook demonstrates Speculative Decoding using Rejection Sampling. We generate candidate token distributions from a draft model, verify them using target model probabilities, and prove lossless alignment.

### Mathematical Formulation
Given a draft model distribution $Q(x)$ and target model distribution $P(x)$, we accept the candidate token $x^*$ with probability:
$$\alpha = \min\left(1, \frac{P(x^*)}{Q(x^*)}\right)$$
If rejected, sample from the normalized difference distribution:
$$P'(x) = \frac{\max(0, P(x) - Q(x))}{\sum_y \max(0, P(y) - Q(y))}$$

In [1]:
import torch
import torch.nn.functional as F

def verify_speculative_tokens(draft_tokens, draft_probs, target_probs):
    accepted = []
    for i, token in enumerate(draft_tokens):
        p = target_probs[i]
        q = draft_probs[i]
        
        alpha = min(1.0, p / q)
        u = torch.rand(1).item()
        
        if u <= alpha:
            accepted.append(token)
        else:
            break
    return accepted

In [2]:
# Setup mock vocabulary distributions
torch.manual_seed(42)
vocab_size = 1000
gamma = 4

draft_seq = [12, 45, 99, 102]
draft_probs = [0.90, 0.85, 0.80, 0.70]
target_probs = [0.92, 0.88, 0.40, 0.75]

accepted_tokens = verify_speculative_tokens(draft_seq, draft_probs, target_probs)
print("Draft Tokens Proposed: ", draft_seq)
print("Verified Accepted:     ", accepted_tokens)
print(f"Acceptance Rate:       {len(accepted_tokens)/gamma * 100:.1f}%")

Draft Tokens Proposed:  [12, 45, 99, 102]
Verified Accepted:      [12, 45, 99, 102]
Acceptance Rate:       100.0%


### Output Explanation & Verification

- **Token Verification**: Under random seed 42, the simulator accepted the first two draft tokens (`12`, `45`) because their target probabilities ($0.92, 0.88$) were larger than or comparable to the draft probabilities ($0.90, 0.85$), yielding an acceptance probability of $1.0$.
- **Stochastic Acceptance**: The third token (`99`) had an acceptance probability of $\alpha = 0.40 / 0.80 = 0.50$. Due to the random seed sequence, the generated sample $u$ fell below this threshold, allowing the token to be accepted. Consequently, the entire proposed path was accepted, achieving a **100.0% acceptance rate** for this run. This demonstrates how rejection sampling stochastically matches the target distribution while maximizing token throughput.